In [37]:
#%pip install pandas==2.2.3
#%pip install git+https://github.com/pydata/pandas-datareader.git
#%pip install yfinance numpy matplotlib
#%pip install fredapi
#%pip install pyarrow
#%pip install scikit-learn

In [38]:
#Imports necesarios para mdescargar los datos y manejar el dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from pandas_datareader import data as pdr 
from fredapi import Fred
from sklearn.feature_selection import mutual_info_regression, f_regression, mutual_info_classif
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer



In [39]:
def imputar_valores_faltantes(dataframe, nombre_columna=None):

    if isinstance(dataframe, pd.Series):
        serie_imputada = dataframe.copy()
    elif isinstance(dataframe, pd.DataFrame):
        if nombre_columna is not None:
            serie_imputada = dataframe[nombre_columna].copy()
        elif dataframe.shape[1] == 1:
            serie_imputada = dataframe.iloc[:, 0].copy()
        else:
            raise ValueError("Debes especificar 'nombre_columna'")
    else:
        raise TypeError("Input debe ser DataFrame o Series")

    serie_imputada = pd.to_numeric(serie_imputada, errors='coerce')

    valores_cero = (serie_imputada == 0)
    valores_nan = serie_imputada.isna()
    total_a_imputar = valores_cero.sum() + valores_nan.sum()

    serie_imputada[valores_cero] = np.nan

    if total_a_imputar == 0:
        return serie_imputada

    metodo1 = serie_imputada.interpolate(method='linear', limit_direction='both')

    media_3m = serie_imputada.rolling(window=3, center=True, min_periods=1).mean()
    media_6m = serie_imputada.rolling(window=6, center=True, min_periods=1).mean()
    metodo2 = media_3m * 0.7 + media_6m * 0.3

    ffill = serie_imputada.ffill()
    bfill = serie_imputada.bfill()
    metodo3 = ffill.fillna(bfill)

    mask = serie_imputada.isna()
    valores_nulos = np.where(mask)[0]

    for valor in valores_nulos:
        valores = []
        pesos = []

        if valor < len(metodo1) and not pd.isna(metodo1.iloc[valor]):
            valores.append(metodo1.iloc[valor])
            pesos.append(0.50)

        if valor < len(metodo2) and not pd.isna(metodo2.iloc[valor]):
            valores.append(metodo2.iloc[valor])
            pesos.append(0.30)

        if valor < len(metodo3) and not pd.isna(metodo3.iloc[valor]):
            valores.append(metodo3.iloc[valor])
            pesos.append(0.20)
 
        if valores:
            pesos_normalizados = np.array(pesos) / np.sum(pesos)
            valor_imputado = np.average(valores, weights=pesos_normalizados)
            serie_imputada.iloc[valor] = valor_imputado

    if serie_imputada.isna().any():
        serie_imputada = serie_imputada.interpolate(method='linear', limit_direction='both')

    if serie_imputada.isna().any():
        serie_imputada = serie_imputada.ffill()
        serie_imputada = serie_imputada.bfill()

    return serie_imputada

In [40]:
def mutual_info_like_r(X, y, n_bins=10, random_state=42):
    """Calcula MI simulando el comportamiento de R (discretización + entropía discreta)"""
    
    # Discretizar X (todas las variables)
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='uniform')
    X_discrete = discretizer.fit_transform(X)
    X_discrete = pd.DataFrame(X_discrete, columns=X.columns, index=X.index)
    
    # Discretizar y (target)
    y_discrete = pd.cut(y, bins=n_bins, labels=False)
    
    # Calcular MI con variables discretas
    mi_scores = mutual_info_classif(X_discrete, y_discrete, random_state=random_state)
    
    return mi_scores

def seleccion_variables(df, target='target', 
                        umbral_mi_percentil=50,
                        umbral_correlacion=0.80,
                        n_bins=10,
                        random_state=42):
    
    # ============================================
    # 1. VARIABLES FORZADAS
    # ============================================
    variables_forzadas = [
        'sp500_return', 'vix', 'tipos_fed', 
        'curva10Y2Y', 'inflacion_usa', 'activo_return'
    ]
    
    # ============================================
    # 2. PREPARACIÓN
    # ============================================
    X = df.select_dtypes(include=[np.number]).drop(columns=[target])
    y = df[target]
    
    print("="*60)
    print("INICIANDO SELECCIÓN DE VARIABLES")
    print("="*60)
    print(f"Variables iniciales: {len(X.columns)}")
    
    # ============================================
    # 3. CALCULAR MI COMO EN R (discretización)
    # ============================================
    from sklearn.preprocessing import KBinsDiscretizer
    from sklearn.feature_selection import mutual_info_classif
    
    # Discretizar X
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='uniform')
    X_discrete = discretizer.fit_transform(X)
    X_discrete = pd.DataFrame(X_discrete, columns=X.columns, index=X.index)
    
    # Discretizar y (target)
    y_discrete = pd.cut(y, bins=n_bins, labels=False)
    
    # Calcular MI con variables discretas
    mi_scores = mutual_info_classif(X_discrete, y_discrete, random_state=random_state)
    
    mi_df = pd.DataFrame({
        'variable': X.columns,
        'mi_score': mi_scores
    }).sort_values('mi_score', ascending=False)
    
    print(f"\n1. Información Mutua (Top 15) - MÉTODO R (discretización):")
    for i, row in mi_df.head(15).iterrows():
        print(f"   {row['variable']:25s} {row['mi_score']:.6f}")
    
    # ============================================
    # 4. SELECCIONAR VARIABLES SIN CORRELACIÓN
    # ============================================
    print(f"\n2. Seleccionando variables sin correlación > {umbral_correlacion}...")
    
    mi_dict = dict(zip(mi_df['variable'], mi_df['mi_score']))
    vars_ordenadas = mi_df['variable'].tolist()
    
    variables_sin_corr = []
    matriz_corr = X.corr().abs()
    
    for var in vars_ordenadas:
        es_redundante = False
        
        for selected in variables_sin_corr:
            corr_val = matriz_corr.loc[var, selected]
            if corr_val > umbral_correlacion:
                es_redundante = True
                print(f"   Eliminando '{var}' (correlación {corr_val:.3f} con '{selected}')")
                break
        
        if not es_redundante:
            variables_sin_corr.append(var)
    
    print(f"\n   Variables sin correlación ({len(variables_sin_corr)}):")
    for var in variables_sin_corr[:10]:
        print(f"      {var:25s} (MI: {mi_dict[var]:.6f})")
    
    # ============================================
    # 5. CALCULAR UMBRAL
    # ============================================
    n_vars = len(variables_sin_corr)
    umbral_index = int(np.ceil(n_vars * umbral_mi_percentil / 100))
    
    print(f"\n3. Cálculo del umbral:")
    print(f"   Nº variables sin correlación: {n_vars}")
    print(f"   Percentil {umbral_mi_percentil}% → nos quedamos con las {umbral_index} primeras")
    
    vars_seleccionadas = variables_sin_corr[:umbral_index]
    
    print(f"\n4. Variables seleccionadas por MI ({len(vars_seleccionadas)}):")
    for var in vars_seleccionadas:
        print(f"      {var:25s} (MI: {mi_dict[var]:.6f})")
    
    # ============================================
    # 6. AÑADIR VARIABLES FORZADAS
    # ============================================
    for var in variables_forzadas:
        if var in X.columns and var not in vars_seleccionadas:
            print(f"   Añadiendo forzada: '{var}' (MI: {mi_dict.get(var, 0):.6f})")
            vars_seleccionadas.append(var)
    
    # ============================================
    # 7. RESULTADO FINAL
    # ============================================
    print("\n" + "="*60)
    print("RESUMEN FINAL")
    print("="*60)
    print(f"✓ Variables iniciales: {len(X.columns)}")
    print(f"✓ Variables finales: {len(vars_seleccionadas)}")
    
    print(f"\n🎯 Variables seleccionadas:")
    for i, var in enumerate(vars_seleccionadas, 1):
        mi_val = mi_dict.get(var, 0)
        es_forzada = " [FORZADA]" if var in variables_forzadas else ""
        print(f"   {i:2d}. {var:30s} (MI: {mi_val:.6f}){es_forzada}")
    
    # ============================================
    # 8. CREAR DATASET FINAL
    # ============================================
    dataset_final = df[vars_seleccionadas + [target]].copy()
    
    return dataset_final

In [41]:
dataset = pd.DataFrame() #dataset global

inicio = '2001-12-01' #fecha en la que empezamos a descargar datos
fin = '2026-03-01' #fecga en la que dejamos de descargar datos

sp500 = yf.download('^GSPC', start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierredel sp500
sp500 = sp500.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes
sp500["sp500_return"] = sp500.pct_change() #Calculamos el retorno mensual del sp500 y lo guardamos en una nueva columna
sp500 = sp500.dropna() #Eliminamos los valores nulos
sp500 = sp500[["sp500_return"]]
#sp500.info()

sp500 = sp500.reset_index()
dataset = sp500 #Asignamos el dataset global al dataset del sp500, que es el que vamos a ir ampliando con las demás variables
dataset.head()

[*********************100%***********************]  1 of 1 completed

Ticker,Date,sp500_return
0,2002-01-31,-0.015574
1,2002-02-28,-0.020766
2,2002-03-31,0.036739
3,2002-04-30,-0.061418
4,2002-05-31,-0.009081


In [42]:
eurodollar = yf.download('EURUSD=X', start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierre del eurodolar
#eurodollar.info()
eurodollar = eurodollar.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes
eurodollar = eurodollar.rename(columns={"Close": "EUR/USD"})
eurodollar = eurodollar.reset_index()

dataset = pd.merge(dataset,eurodollar, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del eurodolar, uniendo por la columna "Date" y quedándonos solo con las filas que tengan fecha en ambos datasets

dataset.head()

[*********************100%***********************]  1 of 1 completed


Ticker,Date,sp500_return,EURUSD=X
0,2003-12-31,0.050766,1.259002
1,2004-01-31,0.017276,1.245206
2,2004-02-29,0.012209,1.253007
3,2004-03-31,-0.016359,1.231300
4,2004-04-30,-0.016791,1.198294


In [43]:
vix = yf.download('^VIX', start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierre del vix
vix.info()
vix = vix.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el vix ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
vix = vix.rename(columns={"Close":"vix"})
vix = vix.reset_index()
#vix.head()

dataset = pd.merge(dataset, vix, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del vix, uniendo por la columna "Date" y quedándonos solo con las filas que tengan fecha en ambos datasets
dataset.head()

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 291 entries, 2001-12-01 to 2026-02-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ^VIX    291 non-null    float64
dtypes: float64(1)
memory usage: 4.5 KB


Ticker,Date,sp500_return,EURUSD=X,^VIX
0,2003-12-31,0.050766,1.259002,18.309999
1,2004-01-31,0.017276,1.245206,16.629999
2,2004-02-29,0.012209,1.253007,14.550000
3,2004-03-31,-0.016359,1.231300,16.740000
4,2004-04-30,-0.016791,1.198294,17.190001


In [44]:
petroleo = yf.download('CL=F', start=inicio, end=fin, interval='1mo') 
petroleo.info()
petroleo.columns = petroleo.columns.get_level_values(0) #El dataset del petroleo tiene un multiindex en las columnas, con el nombre del ticker y el nombre de la variable, por lo que tenemos que eliminar el nivel del ticker para quedarnos solo con el nombre de la variable
petroleo = petroleo.resample("ME").last()
petroleo = petroleo[['Close']].copy()
petroleo['Close'] = imputar_valores_faltantes(petroleo, 'Close')
petroleo["petroleo_return"] = petroleo["Close"].pct_change()
petroleo = petroleo[["petroleo_return"]]
petroleo = petroleo.reset_index()

dataset = pd.merge(dataset, petroleo, on="Date", how="inner")

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 249 entries, 2001-12-01 to 2026-01-01
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, CL=F)   249 non-null    float64
 1   (High, CL=F)    249 non-null    float64
 2   (Low, CL=F)     249 non-null    float64
 3   (Open, CL=F)    249 non-null    float64
 4   (Volume, CL=F)  249 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 11.7 KB


In [45]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             266 non-null    datetime64[ns]
 1   sp500_return     266 non-null    float64       
 2   EURUSD=X         266 non-null    float64       
 3   ^VIX             266 non-null    float64       
 4   petroleo_return  266 non-null    float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 10.5 KB


In [46]:
oro = yf.download('GC=F', start=inicio, end=fin, interval='1mo')
oro.info()  
oro.columns = oro.columns.get_level_values(0)
oro = oro.resample("ME").last()
oro = oro[['Close']].copy()
oro['Close'] = imputar_valores_faltantes(oro, 'Close')
oro["oro_return"] = oro["Close"].pct_change()
oro = oro[["oro_return"]]
oro = oro.reset_index()

dataset = pd.merge(dataset, oro, on="Date", how="inner")

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 249 entries, 2001-12-01 to 2026-01-01
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, GC=F)   249 non-null    float64
 1   (High, GC=F)    249 non-null    float64
 2   (Low, GC=F)     249 non-null    float64
 3   (Open, GC=F)    249 non-null    float64
 4   (Volume, GC=F)  249 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 11.7 KB


In [47]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             266 non-null    datetime64[ns]
 1   sp500_return     266 non-null    float64       
 2   EURUSD=X         266 non-null    float64       
 3   ^VIX             266 non-null    float64       
 4   petroleo_return  266 non-null    float64       
 5   oro_return       266 non-null    float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 12.6 KB


In [48]:
fred = Fred(api_key="af59516007957e0c89174b4d426736ca")

tipos_fed = fred.get_series("FEDFUNDS", observation_start=inicio, observation_end=fin)
tipos_fed = tipos_fed.to_frame()

tipos_fed.info()
tipos_fed = tipos_fed.resample("ME").last()
tipos_fed = tipos_fed.rename(columns={tipos_fed.columns[0]: "tipos_fed"})
tipos_fed = tipos_fed.reset_index()
tipos_fed = tipos_fed.rename(columns={"index":"Date"})

dataset = pd.merge(dataset, tipos_fed, on="Date", how="inner")

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 292 entries, 2001-12-01 to 2026-03-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       292 non-null    float64
dtypes: float64(1)
memory usage: 4.6 KB


In [49]:
dataset.head()

,Date,sp500_return,EURUSD=X,^VIX,petroleo_return,oro_return,tipos_fed
0,2003-12-31,0.050766,1.259002,18.309999,0.069385,0.047631,0.98
1,2004-01-31,0.017276,1.245206,16.629999,0.016298,-0.032475,1.00
2,2004-02-29,0.012209,1.253007,14.550000,0.031217,0.022960,1.01
3,2004-03-31,-0.016359,1.231300,16.740000,0.049243,0.038561,1.00
4,2004-04-30,-0.016791,1.198294,17.190001,0.045302,-0.094313,1.00


In [50]:
ipc_usa = fred.get_series("CPIAUCSL", observation_start=inicio, observation_end=fin) #Descargamos los datos del IPC de Estados Unidos desde la base de datos de FRED, utilizando el código CPIAUCSL que corresponde al IPC de Estados Unidos, y especificando el rango de fechas que queremos descargar
ipc_usa = ipc_usa.to_frame() 

ipc_usa.info()
ipc_usa = ipc_usa.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el IPC de Estados Unidos ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
ipc_usa["inflacion_usa"] = ipc_usa.iloc[:, 0].pct_change(12) #Calculamos la inflación interanual de Estados Unidos usando iloc para acceder a la primera columna (evita problemas de nombres)
ipc_usa = ipc_usa[["inflacion_usa"]]
ipc_usa = ipc_usa.reset_index()
ipc_usa = ipc_usa.rename(columns={"index": "Date"}) #CORREGIDO: la columna de fecha después de reset_index() se llama "index", no "DATE"

dataset = pd.merge(dataset, ipc_usa, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del IPC de Estados Unidos, uniendo por la columna Date

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 292 entries, 2001-12-01 to 2026-03-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       291 non-null    float64
dtypes: float64(1)
memory usage: 4.6 KB


C:\Users\34620\AppData\Local\Temp\ipykernel_34540\3519472962.py:6: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ipc_usa["inflacion_usa"] = ipc_usa.iloc[:, 0].pct_change(12) #Calculamos la inflación interanual de Estados Unidos usando iloc para acceder a la primera columna (evita problemas de nombres)


In [51]:
dataset.head()

,Date,sp500_return,EURUSD=X,^VIX,petroleo_return,oro_return,tipos_fed,inflacion_usa
0,2003-12-31,0.050766,1.259002,18.309999,0.069385,0.047631,0.98,0.020352
1,2004-01-31,0.017276,1.245206,16.629999,0.016298,-0.032475,1.00,0.020263
2,2004-02-29,0.012209,1.253007,14.550000,0.031217,0.022960,1.01,0.016885
3,2004-03-31,-0.016359,1.231300,16.740000,0.049243,0.038561,1.00,0.017401
4,2004-04-30,-0.016791,1.198294,17.190001,0.045302,-0.094313,1.00,0.022926


In [52]:
ipc_eurozona = fred.get_series("CP0000EZ19M086NEST", observation_start=inicio, observation_end=fin)
ipc_eurozona = ipc_eurozona.to_frame()

ipc_eurozona.info()
ipc_eurozona = ipc_eurozona.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el IPC de la eurozona ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
ipc_eurozona["inflation_eu"] = ipc_eurozona.iloc[:, 0].pct_change(12) #Calculamos la inflación interanual de la eurozona usando iloc para acceder a la primera columna 
ipc_eurozona = ipc_eurozona[["inflation_eu"]]
ipc_eurozona = ipc_eurozona.reset_index()
ipc_eurozona = ipc_eurozona.rename(columns={"index": "Date"}) #CORREGIDO: la columna de fecha después de reset_index() se llama "index", no "DATE"

dataset = pd.merge(dataset, ipc_eurozona, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del IPC de la eurozona, uniendo por la columna Date 

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 289 entries, 2001-12-01 to 2025-12-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       289 non-null    float64
dtypes: float64(1)
memory usage: 4.5 KB


In [53]:
dataset.head()

,Date,sp500_return,EURUSD=X,^VIX,petroleo_return,oro_return,tipos_fed,inflacion_usa,inflation_eu
0,2003-12-31,0.050766,1.259002,18.309999,0.069385,0.047631,0.98,0.020352,0.020207
1,2004-01-31,0.017276,1.245206,16.629999,0.016298,-0.032475,1.00,0.020263,0.018343
2,2004-02-29,0.012209,1.253007,14.550000,0.031217,0.022960,1.01,0.016885,0.016654
3,2004-03-31,-0.016359,1.231300,16.740000,0.049243,0.038561,1.00,0.017401,0.017175
4,2004-04-30,-0.016791,1.198294,17.190001,0.045302,-0.094313,1.00,0.022926,0.020851


In [54]:
desempleoUSA = fred.get_series("UNRATE", observation_start=inicio, observation_end=fin)
desempleoUSA = desempleoUSA.to_frame()

desempleoUSA.info()
desempleoUSA = desempleoUSA.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el desempleo de Estados Unidos ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
desempleoUSA = desempleoUSA.rename(columns={desempleoUSA.columns[0]: "desempleo"})
desempleoUSA = desempleoUSA.reset_index()
desempleoUSA = desempleoUSA.rename(columns={"index": "Date"})

dataset = pd.merge(dataset, desempleoUSA, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del desempleo de Estados Unidos, uniendo por la columna Date

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 292 entries, 2001-12-01 to 2026-03-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       291 non-null    float64
dtypes: float64(1)
memory usage: 4.6 KB


In [55]:
dataset.tail()

,Date,sp500_return,EURUSD=X,^VIX,petroleo_return,oro_return,tipos_fed,inflacion_usa,inflation_eu,desempleo
260,2025-08-31,0.019066,1.168634,15.360000,-0.075801,0.054810,4.33,0.029386,0.020289,4.3
261,2025-09-30,0.035324,1.173144,16.280001,-0.025621,0.105680,4.22,0.030226,0.022124,4.4
262,2025-10-31,0.022687,1.157247,17.440001,-0.022286,0.036815,4.09,0.027291,0.020868,NaN
263,2025-11-30,0.001300,1.159689,16.350000,-0.039849,0.059289,3.88,0.026964,0.021093,4.5
264,2025-12-31,-0.000524,1.174729,14.950000,-0.019300,0.025437,3.72,0.026533,0.019366,4.4


In [56]:
tipos_ecb = fred.get_series("ECBDFR") #Descargamos los datos de los tipos de interés de la ECB desde la base de datos de FRED
tipos_ecb = tipos_ecb.to_frame() 

tipos_ecb.info()
tipos_ecb = tipos_ecb.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque los tipos de interés de la ECB ya se publican con datos mensuales, pero lo hacemos por consistencia con el resto de variables
tipos_ecb = tipos_ecb.rename(columns={tipos_ecb.columns[0]: "tipos_ecb"})
tipos_ecb = tipos_ecb.reset_index()
tipos_ecb = tipos_ecb.rename(columns={"index":"Date"}) #Renombramos la columna index a Date


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9969 entries, 1999-01-01 to 2026-04-17
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       9969 non-null   float64
dtypes: float64(1)
memory usage: 155.8 KB


In [57]:
dataset = pd.merge(dataset, tipos_ecb, on="Date", how="inner")

In [58]:
curva10Y2 = fred.get_series("T10Y2Y") #Descargamos los datos de la curva de tipos de interés a 10 años menos la curva de tipos de interés a 2 años desde la base de datos de FRED
curva10Y2 = curva10Y2.to_frame()

curva10Y2.info()
curva10Y2 = curva10Y2.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el IPC de la eurozona ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
curva10Y2 = curva10Y2.rename(columns={curva10Y2.columns[0]: "curva10Y2Y"})
curva10Y2 = curva10Y2.reset_index()
curva10Y2 = curva10Y2.rename(columns={"index":"Date"})


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 13014 entries, 1976-06-01 to 2026-04-17
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       12466 non-null  float64
dtypes: float64(1)
memory usage: 203.3 KB


In [59]:
dataset = pd.merge(dataset, curva10Y2, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del IPC de la eurozona, uniendo por la columna Date 

In [60]:
dataset = dataset.rename(columns={"0_x":"tipos_ecb"}) #Renombramos la columna 0_x a tipos_ecb
dataset = dataset.rename(columns={"0_y":"curva10Y2Y"}) #Renombramos la columna 0_y a curva10Y2Y
dataset.tail()
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             265 non-null    datetime64[ns]
 1   sp500_return     265 non-null    float64       
 2   EURUSD=X         265 non-null    float64       
 3   ^VIX             265 non-null    float64       
 4   petroleo_return  265 non-null    float64       
 5   oro_return       265 non-null    float64       
 6   tipos_fed        265 non-null    float64       
 7   inflacion_usa    265 non-null    float64       
 8   inflation_eu     265 non-null    float64       
 9   desempleo        264 non-null    float64       
 10  tipos_ecb        265 non-null    float64       
 11  curva10Y2Y       265 non-null    float64       
dtypes: datetime64[ns](1), float64(11)
memory usage: 25.0 KB


In [61]:
activo = 'AAPL'

datos_activo = yf.download(activo, start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierre del activo que queramos analizar
datos_activo.info()
datos_activo = datos_activo.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque los datos del activo ya se publican con datos mensuales, pero lo hacemos por consistencia con el resto de variables    
#datos_activo = datos_activo.to_frame()
datos_activo = datos_activo.rename(columns={activo:"precio"})
datos_activo["activo_return"] = datos_activo["precio"].pct_change() #Calculamos el retorno mensual del activo y lo guardamos en una nueva columna

datos_activo['media_movil_3m'] = datos_activo["precio"].rolling(3).mean() #Calculamos la media móvil de 3 meses del precio del activo y la guardamos en una nueva columna
datos_activo['media_movil_6m'] = datos_activo["precio"].rolling(6).mean() #Calculamos la media móvil de 6 meses del precio del activo y la guardamos en una nueva columna
datos_activo['media_movil_12m'] = datos_activo["precio"].rolling(12).mean() #Calculamos la media móvil de 12 meses del precio del activo y la guardamos en una nueva columna

datos_activo['volatilidad_3m'] = datos_activo["activo_return"].rolling(3).std() #Calculamos la volatilidad de 3 meses del precio del activo y la guardamos en una nueva columna
datos_activo['volatilidad_6m'] = datos_activo["activo_return"].rolling(6).std() #Calculamos la volatilidad de 6 meses del precio del activo y la guardamos en una nueva columna

datos_activo['momentum_3m'] = datos_activo["precio"].rolling(3).sum() #Calculamos el momentum de 3 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['momentum_6m'] = datos_activo["precio"].rolling(6).sum() #Calculamos el momentum de 6 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['momentum_12m'] = datos_activo["precio"].rolling(12).sum() #Calculamos el momentum de 12 meses del precio del activo y lo guardamos en una nueva columna

datos_activo['lag_1'] = datos_activo["activo_return"].shift(1) #Calculamos el lag de 1 mes del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_2'] = datos_activo["activo_return"].shift(2) #Calculamos el lag de 2 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_3'] = datos_activo["activo_return"].shift(3) #Calculamos el lag de 3 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_6'] = datos_activo["activo_return"].shift(6) #Calculamos el lag de 6 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_12'] = datos_activo["activo_return"].shift(12) #Calculamos el lag de 12 meses del precio del activo y lo guardamos en una nueva columna
datos_activo.info()

datos_activo = datos_activo.reset_index()

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 291 entries, 2001-12-01 to 2026-02-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    291 non-null    float64
dtypes: float64(1)
memory usage: 4.5 KB
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 291 entries, 2001-12-31 to 2026-02-28
Freq: ME
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   precio           291 non-null    float64
 1   activo_return    290 non-null    float64
 2   media_movil_3m   289 non-null    float64
 3   media_movil_6m   286 non-null    float64
 4   media_movil_12m  280 non-null    float64
 5   volatilidad_3m   288 non-null    float64
 6   volatilidad_6m   285 non-null    float64
 7   momentum_3m      289 non-null    float64
 8   momentum_6m      286 non-null    float64
 9   momentum_12m     280 non-null    float64
 10  lag_1            289 non-null    flo

In [62]:
dataset = pd.merge(dataset, datos_activo, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del activo, uniendo por la columna Date

In [63]:
dataset = dataset.rename(columns={
    "EURUSD=X": "eurusd",
    "^VIX": "vix"
}) #Renombramos las columnas del eurodolar y del vix para que tengan nombres más cortos y fáciles de manejar

#dataset.info()
#dataset.iloc[258:]
#dataset.tail()

dataset['desempleo'] = imputar_valores_faltantes(dataset, 'desempleo')
dataset['activo_return'] = imputar_valores_faltantes(dataset, 'activo_return')
dataset['media_movil_3m'] = imputar_valores_faltantes(dataset, 'media_movil_3m')
dataset['media_movil_6m'] = imputar_valores_faltantes(dataset, 'media_movil_6m')
dataset['media_movil_12m'] = imputar_valores_faltantes(dataset, 'media_movil_12m')
dataset['media_movil_3m'] = imputar_valores_faltantes(dataset, 'media_movil_3m')
dataset['volatilidad_3m'] = imputar_valores_faltantes(dataset, 'volatilidad_3m')
dataset['volatilidad_6m'] = imputar_valores_faltantes(dataset, 'volatilidad_6m')
dataset['momentum_3m'] = imputar_valores_faltantes(dataset, 'momentum_3m')
dataset['momentum_6m'] = imputar_valores_faltantes(dataset, 'momentum_6m')
dataset['momentum_12m'] = imputar_valores_faltantes(dataset, 'momentum_12m')
dataset['lag_1'] = imputar_valores_faltantes(dataset, 'lag_1')
dataset['lag_2'] = imputar_valores_faltantes(dataset, 'lag_2')
dataset['lag_3'] = imputar_valores_faltantes(dataset, 'lag_3')
dataset['lag_6'] = imputar_valores_faltantes(dataset, 'lag_6')
dataset['lag_12'] = imputar_valores_faltantes(dataset, 'lag_12')

dataset['desempleo'] = dataset['desempleo'].round(2)
dataset.isnull().sum()
#dataset.iloc[258:] 

dataset["target"] = dataset["activo_return"].shift(-1) #Creamos la columna target, que es el retorno del activo en el mes siguiente
dataset['target'] = imputar_valores_faltantes(dataset, 'target')

In [64]:
#dataset['target'].isnull()
#dataset.iloc[:6]
print(dataset.columns)

#Vamos a ver los valores de cada columna, para asi ver posibles anomalias
print(dataset['petroleo_return'].value_counts()) 
print(dataset['oro_return'].value_counts()) 

mask = (dataset['oro_return'] == 0) | (dataset['petroleo_return'] == 0)

#print(dataset[mask].head()) #Mostramos las filas donde el retorno del oro o del petroleo es 0, para ver si hay alguna anomalía en esas fechas
#Una vez arreglado el problema de los 0, vemos que el oro y petroleo return, ya están bien

#print(dataset['petroleo_return'].value_counts()) 
# #dataset['oro_return'].value_counts() 

#dataset[['Date','activo_return', 'target']]


#dataset.dtypes

dataset.sort_values(by="Date", ascending=True, inplace=True) #Ordenamos el dataset por la columna Date para asegurarnos de que los datos están en orden cronológico
dataset.head()

Index(['Date', 'sp500_return', 'eurusd', 'vix', 'petroleo_return',
       'oro_return', 'tipos_fed', 'inflacion_usa', 'inflation_eu', 'desempleo',
       'tipos_ecb', 'curva10Y2Y', 'precio', 'activo_return', 'media_movil_3m',
       'media_movil_6m', 'media_movil_12m', 'volatilidad_3m', 'volatilidad_6m',
       'momentum_3m', 'momentum_6m', 'momentum_12m', 'lag_1', 'lag_2', 'lag_3',
       'lag_6', 'lag_12', 'target'],
      dtype='object')
petroleo_return
 0.069385    1
 0.016298    1
 0.031217    1
 0.049243    1
 0.045302    1
            ..
-0.075801    1
-0.025621    1
-0.022286    1
-0.039849    1
-0.019300    1
Name: count, Length: 265, dtype: int64
oro_return
 0.047631    1
-0.032475    1
 0.022960    1
 0.038561    1
-0.094313    1
            ..
 0.054810    1
 0.105680    1
 0.036815    1
 0.059289    1
 0.025437    1
Name: count, Length: 265, dtype: int64


,Date,sp500_return,eurusd,vix,petroleo_return,oro_return,tipos_fed,inflacion_usa,inflation_eu,desempleo,...,volatilidad_6m,momentum_3m,momentum_6m,momentum_12m,lag_1,lag_2,lag_3,lag_6,lag_12,target
0,2003-12-31,0.050766,1.259002,18.309999,0.069385,0.047631,0.98,0.020352,0.020207,5.7,...,0.088742,0.976345,1.941303,3.360652,-0.086500,0.104730,-0.083592,0.061837,-0.075485,0.055685
1,2004-01-31,0.017276,1.245206,16.629999,0.016298,-0.032475,1.00,0.020263,0.018343,5.7,...,0.081350,0.971401,1.963476,3.483500,0.021998,-0.086500,0.104730,0.105983,0.002095,0.060284
2,2004-02-29,0.012209,1.253007,14.550000,0.031217,0.022960,1.01,0.016885,0.016654,5.6,...,0.079723,1.016495,1.983101,3.616985,0.055685,0.021998,-0.086500,0.072580,0.045264,0.130434
3,2004-03-31,-0.016359,1.231300,16.740000,0.049243,0.038561,1.00,0.017401,0.017175,5.8,...,0.076144,1.101440,2.077784,3.810246,0.060284,0.055685,0.021998,-0.083592,-0.057962,-0.046597
4,2004-04-30,-0.016791,1.198294,17.190001,0.045302,-0.094313,1.00,0.022926,0.020851,5.6,...,0.078530,1.149680,2.121081,3.983432,0.130434,0.060284,0.055685,0.104730,0.005659,0.088440


In [65]:
dataset_optmizado = seleccion_variables(dataset, target='target')

variables_seleccionadas = [col for col in dataset_optmizado.columns if col != 'target']
variables_macro_seleccionadas = [v for v in variables_seleccionadas if v in ['tipos_fed', 'inflacion_usa', 'vix', 'desempleo', 'curva10Y2Y']]

dataset_dl = dataset_optmizado.copy() #Creamos una copia del dataset para usarlo en el modelo de deep learning, asi mantenemos el dataset original por si queremos hacer alguna otra cosa con él
dataset_ml = dataset_optmizado.copy() #Creamos una copia del dataset para usarlo en el modelo de machine learning, asi mantenemos el dataset original por si queremos hacer alguna otra cosa con él

if 'tipos_fed' in variables_macro_seleccionadas:
    dataset_ml['tipos_fed_lag1'] = dataset_ml['tipos_fed'].shift(1)
    dataset_ml['tipos_fed_lag3'] = dataset_ml['tipos_fed'].shift(3)
    dataset_ml['tipos_fed_lag6'] = dataset_ml['tipos_fed'].shift(6)


if 'inflacion_usa' in variables_macro_seleccionadas:
    dataset_ml['inflacion_usa_lag1'] = dataset_ml['inflacion_usa'].shift(1)
    dataset_ml['inflacion_usa_lag3'] = dataset_ml['inflacion_usa'].shift(3)
    dataset_ml['inflacion_usa_lag6'] = dataset_ml['inflacion_usa'].shift(6)
   

if 'vix' in variables_macro_seleccionadas:
    dataset_ml['vix_lag1'] = dataset_ml['vix'].shift(1)
    dataset_ml['vix_lag2'] = dataset_ml['vix'].shift(2)
    dataset_ml['vix_lag3'] = dataset_ml['vix'].shift(3)
   

if 'desempleo' in variables_macro_seleccionadas:
    dataset_ml['desempleo_lag1'] = dataset_ml['desempleo'].shift(1)
    dataset_ml['desempleo_lag3'] = dataset_ml['desempleo'].shift(3)
   

if 'curva10Y2Y' in variables_macro_seleccionadas:
    dataset_ml['curva10Y2Y_lag1'] = dataset_ml['curva10Y2Y'].shift(1)
    dataset_ml['curva10Y2Y_lag3'] = dataset_ml['curva10Y2Y'].shift(3)

for col in dataset_ml.columns:
    if dataset_ml[col].isnull().any():
        dataset_ml[col] = imputar_valores_faltantes(dataset_ml, col)


INICIANDO SELECCIÓN DE VARIABLES
Variables iniciales: 26

1. Información Mutua (Top 15) - MÉTODO R (discretización):
   media_movil_6m            0.152826
   lag_3                     0.128594
   vix                       0.123089
   momentum_12m              0.105900
   oro_return                0.074940
   eurusd                    0.072449
   media_movil_3m            0.071333
   lag_12                    0.069918
   lag_6                     0.062796
   volatilidad_3m            0.058803
   momentum_3m               0.056557
   inflacion_usa             0.050691
   inflation_eu              0.040307
   momentum_6m               0.034097
   sp500_return              0.034035

2. Seleccionando variables sin correlación > 0.8...
   Eliminando 'momentum_12m' (correlación 0.997 con 'media_movil_6m')
   Eliminando 'media_movil_3m' (correlación 0.998 con 'media_movil_6m')
   Eliminando 'momentum_3m' (correlación 0.998 con 'media_movil_6m')
   Eliminando 'inflation_eu' (correlación 0.864 c

In [66]:
#print(dataset_ml.info())
#print(dataset_ml.isnull().sum())

In [67]:
#dataset_ml.head()

In [68]:
dataset.head()

,Date,sp500_return,eurusd,vix,petroleo_return,oro_return,tipos_fed,inflacion_usa,inflation_eu,desempleo,...,volatilidad_6m,momentum_3m,momentum_6m,momentum_12m,lag_1,lag_2,lag_3,lag_6,lag_12,target
0,2003-12-31,0.050766,1.259002,18.309999,0.069385,0.047631,0.98,0.020352,0.020207,5.7,...,0.088742,0.976345,1.941303,3.360652,-0.086500,0.104730,-0.083592,0.061837,-0.075485,0.055685
1,2004-01-31,0.017276,1.245206,16.629999,0.016298,-0.032475,1.00,0.020263,0.018343,5.7,...,0.081350,0.971401,1.963476,3.483500,0.021998,-0.086500,0.104730,0.105983,0.002095,0.060284
2,2004-02-29,0.012209,1.253007,14.550000,0.031217,0.022960,1.01,0.016885,0.016654,5.6,...,0.079723,1.016495,1.983101,3.616985,0.055685,0.021998,-0.086500,0.072580,0.045264,0.130434
3,2004-03-31,-0.016359,1.231300,16.740000,0.049243,0.038561,1.00,0.017401,0.017175,5.8,...,0.076144,1.101440,2.077784,3.810246,0.060284,0.055685,0.021998,-0.083592,-0.057962,-0.046597
4,2004-04-30,-0.016791,1.198294,17.190001,0.045302,-0.094313,1.00,0.022926,0.020851,5.6,...,0.078530,1.149680,2.121081,3.983432,0.130434,0.060284,0.055685,0.104730,0.005659,0.088440


In [69]:
dataset_dl.to_csv('dataset_dl.csv', index=False)
dataset_ml.to_csv('dataset_ml.csv', index=False)